# Scikit-learn Functions Mastery Guide

## Quantitative Research: Pipelines, PCA, Clustering, and Time-Series ML

This notebook is a focused reference for the parts of the scikit-learn ecosystem that are most useful in quantitative research.

It deliberately avoids a catalogue of interchangeable models. The objective is to master the reusable interfaces and research patterns that matter in practice:

- the estimator and transformer APIs;
- leakage-safe preprocessing with `Pipeline`;
- PCA for dimensionality reduction and factor structure;
- KMeans for hard regime clustering;
- Gaussian Mixture Models for probabilistic regime assignment;
- XGBoost through its scikit-learn-compatible interface;
- time-aware fitting, validation, and walk-forward research.

The notebook uses one synthetic market dataset throughout so that every section belongs to the same research workflow.

## 0. Imports and synthetic market data

The data contain several market features with a latent three-regime structure. The latent state is used only to generate the data; the models below do not receive it as an input.

The objective is not to produce a profitable strategy. It is to learn the relevant APIs in a realistic quantitative-research setting.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

RANDOM_STATE = 42
TRADING_DAYS = 252
rng = np.random.default_rng(RANDOM_STATE)


In [2]:
n = 2200
dates = pd.bdate_range("2017-01-02", periods=n)

# Persistent latent regimes: 0 = bear, 1 = neutral, 2 = bull.
transition = np.array([
    [0.965, 0.030, 0.005],
    [0.020, 0.960, 0.020],
    [0.005, 0.030, 0.965],
])

state = np.zeros(n, dtype=int)
state[0] = 1
for t in range(1, n):
    state[t] = rng.choice(3, p=transition[state[t - 1]])

ret_mu = np.array([-0.0010, 0.0001, 0.0009])
ret_sigma = np.array([0.018, 0.008, 0.011])

market_return = rng.normal(ret_mu[state], ret_sigma[state])
realized_vol = pd.Series(market_return).rolling(20).std().bfill().to_numpy() * np.sqrt(TRADING_DAYS)

momentum_20 = pd.Series(market_return).rolling(20).sum().shift(1).fillna(0).to_numpy()
momentum_60 = pd.Series(market_return).rolling(60).sum().shift(1).fillna(0).to_numpy()

credit_spread = (
    np.array([1.8, 1.0, 0.7])[state]
    + rng.normal(0, 0.12, n)
)
term_spread = (
    np.array([-0.2, 0.4, 0.7])[state]
    + rng.normal(0, 0.15, n)
)
volume_z = (
    np.array([1.0, 0.0, 0.3])[state]
    + rng.normal(0, 0.6, n)
)

features = pd.DataFrame({
    "momentum_20": momentum_20,
    "momentum_60": momentum_60,
    "realized_vol": realized_vol,
    "credit_spread": credit_spread,
    "term_spread": term_spread,
    "volume_z": volume_z,
}, index=dates)

target = pd.Series(market_return, index=dates, name="next_return").shift(-1)
latent_regime = pd.Series(state, index=dates, name="latent_regime")

data = features.join(target).dropna()
X = data[features.columns]
y = data["next_return"]

display(X.head())
display(y.head())


,momentum_20,momentum_60,realized_vol,credit_spread,term_spread,volume_z
2017-01-02,0.0,0.0,0.137441,0.862252,0.237164,-0.001405
2017-01-03,0.0,0.0,0.137441,1.010672,0.361252,1.182294
2017-01-04,0.0,0.0,0.137441,0.824256,0.381089,-0.419187
2017-01-05,0.0,0.0,0.137441,0.876732,0.301347,0.518279
2017-01-06,0.0,0.0,0.137441,0.909916,0.409670,-0.287941


2017-01-02   -0.006674
2017-01-03   -0.008505
2017-01-04    0.001894
2017-01-05    0.004908
2017-01-06    0.002760
Freq: B, Name: next_return, dtype: float64

## 1. The scikit-learn interface

Most scikit-learn objects follow a small number of interfaces.

### Estimator

```python
model.fit(X_train, y_train)
```

`fit` estimates parameters from data.

### Predictor

```python
pred = model.predict(X_test)
```

A fitted supervised model maps features to predictions.

### Transformer

```python
transformer.fit(X_train)
X_train_transformed = transformer.transform(X_train)
X_test_transformed = transformer.transform(X_test)
```

A transformer learns a transformation on the training sample and applies the same transformation elsewhere.

In [3]:
split = int(len(X) * 0.75)

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape, X_test.shape)
print(X_train_scaled.shape, X_test_scaled.shape)

# main functions:
# scaler = ..., scaler.fit_transform(X_train), scaler.transform(X_test)


(1649, 6) (550, 6)
(1649, 6) (550, 6)


### Useful fitted attributes

scikit-learn convention:

- constructor arguments are ordinary attributes/parameters;
- quantities learned during `fit` usually end with `_`.

Examples:

```python
scaler.mean_
scaler.scale_
pca.components_
kmeans.cluster_centers_
gmm.means_
```

This convention is useful when inspecting fitted objects.

In [4]:
display(pd.Series(scaler.mean_, index=X.columns, name="training_mean"))
display(pd.Series(scaler.scale_, index=X.columns, name="training_scale"))


momentum_20      0.009094
momentum_60      0.027149
realized_vol     0.183647
credit_spread    1.051070
term_spread      0.389969
volume_z         0.320848
Name: training_mean, dtype: float64

momentum_20      0.058528
momentum_60      0.110853
realized_vol     0.060771
credit_spread    0.433779
term_spread      0.379217
volume_z         0.733578
Name: training_scale, dtype: float64

## 2. Pipeline

`Pipeline` chains preprocessing and estimation into one object.

Its main value is not shorter syntax. Its main value is **correct fitting discipline**.

Consider:

```python
Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(...)),
    ("model", ...)
])
```

Calling:

```python
pipe.fit(X_train, y_train)
```

fits each transformation on `X_train`, transforms the data, and then fits the model.

Calling:

```python
pipe.predict(X_test)
```

applies the already-fitted transformations to `X_test` and then predicts. The preprocessing is **not refitted** on the test data.

This becomes particularly useful during cross-validation because each fold fits its own preprocessing using only that fold's training sample.

In [5]:
pca_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=3)),
])

X_train_pca = pca_pipe.fit_transform(X_train)
X_test_pca = pca_pipe.transform(X_test)

print(X_train_pca.shape)
print(X_test_pca.shape)


(1649, 3)
(550, 3)


### Accessing pipeline steps

Use:

```python
pipe.named_steps["step_name"]
```

to inspect a fitted step.

Pipeline parameters use the convention:

```text
step_name__parameter_name
```

For example:

```python
pca__n_components
model__max_depth
```

This is how tuning utilities address parameters inside a pipeline.

In [6]:
fitted_pca = pca_pipe.named_steps["pca"]

display(
    pd.Series(
        fitted_pca.explained_variance_ratio_,
        index=["PC1", "PC2", "PC3"],
        name="explained_variance_ratio"
    )
)

display(pca_pipe.get_params().keys())

# main functions:
# pipe = Pipeline([
#     ('scaler', StandardScaler()),
#     ('pca', PCA(n_components))
# ]) 
# 
# -> can then call pipe.fit(), pipe.transform() to automate the discipline (scaler -> pca -> model for example)
# 
# for PCA, n_components = int gives the number of components we keep, between 0 and 1 it gives how much variance should be explained
# 
# to access a step -> pipe.named_steps['step_name'] ~ pipe.named_steps['pca']
# to reach a parameter -> pipe.set_params(step_name__parameter_name = value) ~ pipe.set_params(pca__n_components = 0.95)


PC1    0.467776
PC2    0.215436
PC3    0.123136
Name: explained_variance_ratio, dtype: float64

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'scaler', 'pca', 'scaler__copy', 'scaler__with_mean', 'scaler__with_std', 'pca__copy', 'pca__iterated_power', 'pca__n_components', 'pca__n_oversamples', 'pca__power_iteration_normalizer', 'pca__random_state', 'pca__svd_solver', 'pca__tol', 'pca__whiten'])

## 3. PCA

PCA creates orthogonal linear combinations of the original features.

For quantitative research, the important API is small:

```python
pca.fit(X_train)
pca.transform(X_test)
pca.fit_transform(X_train)
pca.explained_variance_ratio_
pca.components_
```

### Scaling

PCA is variance-based. Features measured on larger numerical scales can dominate the components, so standardized inputs are generally appropriate when the raw features have heterogeneous units.

That is why `StandardScaler → PCA` is naturally represented as a pipeline.

In [7]:
pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA()),
])

pca.fit(X_train)

explained = pd.Series(
    pca.named_steps["pca"].explained_variance_ratio_,
    index=[f"PC{i}" for i in range(1, X.shape[1] + 1)]
)

cumulative_explained = explained.cumsum()

display(pd.DataFrame({
    "explained_variance": explained,
    "cumulative_explained_variance": cumulative_explained
}))


,explained_variance,cumulative_explained_variance
PC1,0.467776,0.467776
PC2,0.215436,0.683211
PC3,0.123136,0.806348
PC4,0.106132,0.912480
PC5,0.068511,0.980991
PC6,0.019009,1.000000


In [8]:
loadings = pd.DataFrame(
    pca.named_steps["pca"].components_.T,
    index=X.columns,
    columns=[f"PC{i}" for i in range(1, X.shape[1] + 1)]
)

display(loadings)

# main functions:
# pca.explained_variance_ratio_ -> gives the list of explained variance as a fraction of total variance for each component
# pca.components_ -> gives the coefficients to transform the original column into each component 


,PC1,PC2,PC3,PC4,PC5,PC6
momentum_20,-0.295616,0.649740,-0.075316,0.159105,-0.677477,0.022055
momentum_60,-0.346452,0.582684,-0.067067,0.054923,0.729873,-0.014862
realized_vol,0.406085,0.076502,0.190952,0.885899,0.081842,-0.035558
credit_spread,0.518073,0.251664,-0.346181,-0.159292,0.039884,0.722121
term_spread,-0.504669,-0.252861,0.410882,0.192318,-0.000359,0.689608
volume_z,0.323398,0.324330,0.815292,-0.352833,-0.004597,-0.031778


## 4. KMeans: hard regime clustering

KMeans partitions observations into `n_clusters` groups by minimizing within-cluster squared distance to cluster centroids.

Core API:

```python
kmeans.fit(X_train)
kmeans.predict(X_test)
kmeans.transform(X_test)
kmeans.cluster_centers_
kmeans.inertia_
```

`predict` gives a hard cluster assignment.

`transform` is particularly useful for regime research: it returns the distance of every observation to every centroid. Those distances retain information that a hard label discards.

In [9]:
kmeans_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(
        n_clusters=3,
        n_init=20,
        random_state=RANDOM_STATE
    )),
])

kmeans_pipe.fit(X_train)

train_cluster = pd.Series(
    kmeans_pipe.predict(X_train),
    index=X_train.index,
    name="cluster"
)

test_cluster = pd.Series(
    kmeans_pipe.predict(X_test),
    index=X_test.index,
    name="cluster"
)

display(test_cluster.head())


2023-04-28    0
2023-05-01    0
2023-05-02    0
2023-05-03    0
2023-05-04    0
Freq: B, Name: cluster, dtype: int32

In [10]:
X_test_scaled = kmeans_pipe.named_steps["scaler"].transform(X_test)

centroid_distances = pd.DataFrame(
    kmeans_pipe.named_steps["kmeans"].transform(X_test_scaled),
    index=X_test.index,
    columns=[f"distance_cluster_{i}" for i in range(3)]
)

display(centroid_distances.head())


,distance_cluster_0,distance_cluster_1,distance_cluster_2
2023-04-28,1.320539,4.395246,4.307442
2023-05-01,0.955618,3.844985,3.897169
2023-05-02,0.706994,3.899042,4.021729
2023-05-03,0.989428,4.043487,4.245579
2023-05-04,1.479570,4.834506,4.844503


### Cluster labels have no economic meaning by themselves

Cluster `0`, `1`, and `2` are arbitrary identifiers. They should not automatically be called bear, neutral, and bull.

A common research step is to characterize clusters using variables that were not used to assign economic names mechanically—for example subsequent returns, volatility, or economically interpretable feature averages.

Also note that cluster numbering can change after refitting. A production or walk-forward implementation therefore needs a consistent relabelling rule if the economic regime names matter downstream.

In [11]:
cluster_characteristics = (
    X_train
    .assign(
        cluster=train_cluster,
        next_return=y_train
    )
    .groupby("cluster")
    .agg(
        avg_next_return=("next_return", "mean"),
        avg_realized_vol=("realized_vol", "mean"),
        avg_momentum_20=("momentum_20", "mean"),
        n_obs=("next_return", "count"),
    )
)

display(cluster_characteristics)

# main functions:
# pipe = Pipeline([
#     ("scaler", StandardScaler()),
#     ("kmeans", KMeans(
#         n_clusters=3,
#         n_init=20,
#         random_state=RANDOM_STATE
#     ))
# ]) -> recommended to use different init with KMeans 
# 
# KMeans.predict() gives a hard classification, KMeans.transform() gives distance to each cluster
# 
# Remark: the label does not hold economic value, we need to attribute a state after comparing them 


,avg_next_return,avg_realized_vol,avg_momentum_20,n_obs
cluster,,,,
0,0.000991,0.160209,0.018798,1231
1,-0.000979,0.241125,0.037735,229
2,-0.001375,0.266655,-0.088816,189


## 5. Gaussian Mixture Model: probabilistic regimes

A Gaussian Mixture Model represents the feature distribution as a mixture of Gaussian components.

Unlike KMeans, GMM naturally produces posterior component probabilities.

Core API:

```python
gmm.fit(X_train)
gmm.predict(X_test)
gmm.predict_proba(X_test)
gmm.means_
gmm.covariances_
```

This makes GMM particularly relevant when an observation lies between regimes. Instead of forcing a complete hard switch, the model can return, for example:

```text
Regime 0: 0.08
Regime 1: 0.57
Regime 2: 0.35
```

Those are model-based posterior probabilities under the fitted mixture assumptions.

In [12]:
gmm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("gmm", GaussianMixture(
        n_components=3,
        covariance_type="full",
        n_init=10,
        random_state=RANDOM_STATE
    )),
])

gmm_pipe.fit(X_train)

gmm_test_labels = pd.Series(
    gmm_pipe.predict(X_test),
    index=X_test.index,
    name="gmm_cluster"
)

X_test_scaled = gmm_pipe.named_steps["scaler"].transform(X_test)

gmm_probabilities = pd.DataFrame(
    gmm_pipe.named_steps["gmm"].predict_proba(X_test_scaled),
    index=X_test.index,
    columns=[f"prob_cluster_{i}" for i in range(3)]
)

display(gmm_probabilities.head())


,prob_cluster_0,prob_cluster_1,prob_cluster_2
2023-04-28,0.992239,8.354738e-33,0.007761
2023-05-01,0.995727,1.566011e-28,0.004273
2023-05-02,0.996465,3.444352e-30,0.003535
2023-05-03,0.997509,2.816713e-28,0.002491
2023-05-04,0.997022,2.121021e-51,0.002978


### KMeans distance is not automatically a probability

KMeans gives distances to centroids:

```python
kmeans.transform(X)
```

These can be converted into a custom soft weighting rule, but the result is a modelling choice. It is not a probability implied by KMeans itself.

GMM instead estimates posterior component probabilities directly from its probabilistic mixture model.

This distinction is important when comparing a custom distance-based regime weighting scheme with a probabilistic clustering baseline.

In [13]:
centroid_proba = centroid_distances.copy().pow(-1).pipe(lambda x: x.div(x.sum(axis=1), axis=0))

comparison = pd.concat([
    test_cluster,
    centroid_proba,
    gmm_test_labels,
    gmm_probabilities,
], axis=1)

display(comparison.head(10))

# main functions:
# pipe = Pipeline([
#     ("scaler", StandardScaler()),
#     ("gmm", GaussianMixture(
#         n_components=3,
#         covariance_type="full",
#         n_init=10,
#         random_state=RANDOM_STATE
#     ))
# ])
# 
# -> gmm.predict() gives a hard classification, gmm.predict_proba() gives probabilities for each cluster
# 
# Remark: with KMeans distance, we do 1/dist and normalize but it is a choice, not direct probabilities such as gmm assumes


,cluster,distance_cluster_0,distance_cluster_1,distance_cluster_2,gmm_cluster,prob_cluster_0,prob_cluster_1,prob_cluster_2
2023-04-28,0,0.622270,0.186959,0.190770,0,0.992239,8.354738e-33,0.007761
2023-05-01,0,0.669459,0.166385,0.164157,0,0.995727,1.566011e-28,0.004273
2023-05-02,0,0.736855,0.133610,0.129534,0,0.996465,3.444352e-30,0.003535
2023-05-03,0,0.676706,0.165588,0.157706,0,0.997509,2.816713e-28,0.002491
2023-05-04,0,0.620557,0.189918,0.189526,0,0.997022,2.121021e-51,0.002978
2023-05-05,0,0.721832,0.139912,0.138256,0,0.985791,8.316514e-29,0.014209
2023-05-08,0,0.508141,0.241466,0.250393,0,0.985710,1.126335e-32,0.014290
2023-05-09,0,0.595463,0.194856,0.209681,0,0.990372,7.725617e-39,0.009628
2023-05-10,0,0.547317,0.221885,0.230798,0,0.996895,3.565450e-30,0.003105
2023-05-11,0,0.558392,0.209322,0.232286,0,0.997416,3.832280e-34,0.002584


### Covariance structure

`GaussianMixture` supports several covariance assumptions:

- `"full"`: each component has its own full covariance matrix;
- `"tied"`: all components share one full covariance matrix;
- `"diag"`: each component has its own diagonal covariance matrix;
- `"spherical"`: each component has one variance shared across dimensions.

This is a modelling assumption, not merely a computational setting. In market-regime work, allowing correlations and volatility structures to differ across regimes can be economically meaningful, but more flexible covariance estimation also requires more data.

## 6. XGBoost through the scikit-learn interface

The familiar pattern remains:

```python
model = XGBRegressor(...)
model.fit(X_train, y_train)
pred = model.predict(X_test)
```

The purpose of this section is not to memorize every XGBoost hyperparameter. It is to understand how a model used in quantitative research fits into the same estimator workflow.

In [14]:
if XGBOOST_AVAILABLE:
    xgb = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    xgb.fit(X_train, y_train)
    pred = pd.Series(xgb.predict(X_test), index=X_test.index, name="prediction")

    rmse = np.sqrt(mean_squared_error(y_test, pred))
    print(f"Test RMSE: {rmse:.6f}")
    display(pred.head())
else:
    print("XGBoost is not installed. Install it with: pip install xgboost")


Test RMSE: 0.011605


2023-04-28   -0.001475
2023-05-01    0.000700
2023-05-02    0.000929
2023-05-03    0.000896
2023-05-04    0.006703
Freq: B, Name: prediction, dtype: float32

### Feature importance

Tree-based feature importance can be useful diagnostically, but it should not automatically be interpreted as causal or as a stable economic relationship.

For XGBoost:

```python
model.feature_importances_
```

returns model-specific importance values. Stability across time and out-of-sample behavior matter more than a single fitted ranking.

In [15]:
if XGBOOST_AVAILABLE:
    feature_importance = (
        pd.Series(
            xgb.feature_importances_,
            index=X.columns,
            name="importance"
        )
        .sort_values(ascending=False)
    )

    display(feature_importance)


term_spread      0.181174
credit_spread    0.177280
realized_vol     0.163424
momentum_60      0.160461
volume_z         0.160236
momentum_20      0.157424
Name: importance, dtype: float32

### Early stopping

Early stopping determines the number of boosting rounds using a validation set rather than fixing the effective number of trees in advance.

With XGBoost 3.2.0, `early_stopping_rounds` is specified when creating `XGBRegressor`, while the validation set is passed to `fit` through `eval_set`.

The data therefore have three distinct roles:

- **Training set:** fits the trees.
- **Validation set:** monitors the selected evaluation metric and determines when boosting should stop.
- **Test/OOS set:** remains untouched until the model configuration is finalized and is used only for final evaluation.

Set `n_estimators` sufficiently high to provide an upper bound on the number of boosting rounds. If the validation metric does not improve for `early_stopping_rounds` consecutive rounds, training stops.

After fitting:

```python
model.best_iteration
model.best_score
```

In [16]:
# Keep the final test set untouched.
val_size = int(len(X_train) * 0.20)

X_fit = X_train.iloc[:-val_size]
y_fit = y_train.iloc[:-val_size]

X_val = X_train.iloc[-val_size:]
y_val = y_train.iloc[-val_size:]

xgb_early_stop = XGBRegressor(
    n_estimators=2000,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_early_stop.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f"Best iteration: {xgb_early_stop.best_iteration}")
print(f"Best validation RMSE: {xgb_early_stop.best_score:.6f}")

test_pred = pd.Series(
    xgb_early_stop.predict(X_test),
    index=X_test.index,
    name="prediction"
)

test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
print(f"Test RMSE: {test_rmse:.6f}")
print(f"Test RMSE without early stopping: {rmse:.6f}")

# main functions:
# xgb = XGBRegressor(
#     n_estimators=300,
#     max_depth=3,
#     learning_rate=0.03,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective="reg:squarederror",
#     random_state=RANDOM_STATE,
#     n_jobs=-1,
# ) -> usually goes with hyperparameter tuning from validation. xgb.fit() and xgb.predict() as for other models
# 
# xgb.feature_importances_ gives the importance by feature but depends on fit, OOS and time dynamics are better confirmations
# 
# with early stopping from validation not improving, it can limit the number of trees:

# xgb_early_stop = XGBRegressor(
#     n_estimators=2000,
#     max_depth=3,
#     learning_rate=0.03,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     objective="reg:squarederror",
#     eval_metric="rmse",
#     early_stopping_rounds=50,
#     random_state=RANDOM_STATE,
#     n_jobs=-1,
# )
# 
# xgb_early_stop.fit(
#     X_fit,
#     y_fit,
#     eval_set=[(X_val, y_val)],
#     verbose=False,
# )


Best iteration: 57
Best validation RMSE: 0.010653
Test RMSE: 0.011253
Test RMSE without early stopping: 0.011605


## 7. Time-aware validation

Random cross-validation is generally inappropriate when observations are ordered through time and the research question concerns future prediction.

`TimeSeriesSplit` preserves ordering:

```python
tscv = TimeSeriesSplit(n_splits=5)
```

Each validation fold occurs after its corresponding training fold.

This is useful for generic sklearn-compatible tuning, although a custom walk-forward implementation is often preferable when the research design requires specific expanding windows, validation lengths, embargoes, refit schedules, or a separate untouched OOS sample.

In [17]:
tscv = TimeSeriesSplit(n_splits=5)

folds = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(X), start=1):
    folds.append({
        "fold": fold,
        "train_start": X.index[train_idx[0]],
        "train_end": X.index[train_idx[-1]],
        "val_start": X.index[val_idx[0]],
        "val_end": X.index[val_idx[-1]],
        "n_train": len(train_idx),
        "n_val": len(val_idx),
    })

display(pd.DataFrame(folds))

# main functions:
# tscv = TimeSeriesSplit(n_splits, test_size, gap) -> to split in chronological order, train is expanding here
# then for fold, (train_idx, val_idx) in enumerate(tscv.split(X), start=1)
# by default select test_size = n_samples // (n_splits+1) then goes backward to get the train and test splits checking with gap


,fold,train_start,train_end,val_start,val_end,n_train,n_val
0,1,2017-01-02,2018-05-31,2018-06-01,2019-10-25,369,366
1,2,2017-01-02,2019-10-25,2019-10-28,2021-03-22,735,366
2,3,2017-01-02,2021-03-22,2021-03-23,2022-08-16,1101,366
3,4,2017-01-02,2022-08-16,2022-08-17,2024-01-10,1467,366
4,5,2017-01-02,2024-01-10,2024-01-11,2025-06-05,1833,366


## 8. Parameter search with train, validation, and test sets

Hyperparameter tuning should preserve the distinct role of each dataset:

1. define candidate parameter values;
2. fit each candidate using the training set;
3. evaluate each candidate on the validation set;
4. select the hyperparameters with the best validation performance;
5. evaluate the selected specification on the test set.

The test set must not influence hyperparameter selection. It is used only after the model specification has been chosen.

`ParameterGrid` generates all combinations from a dictionary of candidate parameters while leaving the fitting and evaluation logic under explicit control.

When the model is inside a `Pipeline`, parameters are addressed using:

```text
step_name__parameter_name
```

In [18]:
param_grid = ParameterGrid({
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.05],
})

preprocessor = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
])

results = []

for params in param_grid:
    prep = clone(preprocessor)

    X_train_transformed = prep.fit_transform(X_train)
    X_val_transformed = prep.transform(X_val)

    model = XGBRegressor(
        **params,
        n_estimators=2000,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(
        X_train_transformed,
        y_train,
        eval_set=[(X_val_transformed, y_val)],
        verbose=False,
    )

    val_pred = model.predict(X_val_transformed)

    val_rmse = np.sqrt(
        mean_squared_error(y_val, val_pred)
    )

    results.append({
        "params": params,
        "val_rmse": val_rmse,
        "best_iteration": model.best_iteration,
    })

tuning_results = (
    pd.DataFrame([
        {
            **result["params"],
            "best_iteration": result["best_iteration"],
            "val_rmse": result["val_rmse"],
        }
        for result in results
    ])
    .sort_values("val_rmse")
    .reset_index(drop=True)
)

display(tuning_results)

best_result = min(
    results,
    key=lambda x: x["val_rmse"]
)

best_params = best_result["params"]

print("Best parameters:")
print(best_params)
print(f"Best iteration: {best_result['best_iteration']}")
print(f"Validation RMSE: {best_result['val_rmse']:.6f}")

# Refit the selected specification using the same train/validation structure.
best_preprocessor = clone(preprocessor)

X_train_transformed = best_preprocessor.fit_transform(X_train)
X_val_transformed = best_preprocessor.transform(X_val)
X_test_transformed = best_preprocessor.transform(X_test)

best_model = XGBRegressor(
    **best_params,
    n_estimators=2000,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

best_model.fit(
    X_train_transformed,
    y_train,
    eval_set=[(X_val_transformed, y_val)],
    verbose=False,
)

test_pred = pd.Series(
    best_model.predict(X_test_transformed),
    index=X_test.index,
    name="prediction",
)

test_rmse = np.sqrt(
    mean_squared_error(y_test, test_pred)
)

print(f"Final best iteration: {best_model.best_iteration}")
print(f"Test RMSE: {test_rmse:.6f}")

# main functions:
# param_grid = ParameterGrid({
#     "max_depth": [2, 3],
#     "learning_rate": [0.03, 0.05],
# }) -> create grid of parameters for hyperparameter tuning

# preprocessor = Pipeline([
#     ("scaler", StandardScaler()),
#     ("pca", PCA(n_components=0.95)),
# ]) -> only preprocess to keep the model out with early stopping and grid
# 
# for params in param_grid:
#     prep = clone(preprocessor)

#     X_train_transformed = prep.fit_transform(X_train)
#     X_val_transformed = prep.transform(X_val)

#     model = XGBRegressor(
#         **params,
#         n_estimators=2000,
#         objective="reg:squarederror",
#         eval_metric="rmse",
#         early_stopping_rounds=50,
#         random_state=RANDOM_STATE,
#         n_jobs=-1,
#     )

#     model.fit(
#         X_train_transformed,
#         y_train,
#         eval_set=[(X_val_transformed, y_val)],
#         verbose=False,
#     )
# 
# tuning_results = (
#     pd.DataFrame([
#         {
#             **result["params"],
#             "best_iteration": result["best_iteration"],
#             "val_rmse": result["val_rmse"],
#         }
#         for result in results
#     ])
#     .sort_values("val_rmse")
#     .reset_index(drop=True)
# ) -> store the result and ready to inspect, especially regarding stability
# 
# 
# best_result = min(
#     results,
#     key=lambda x: x["val_rmse"]
# )
# best_params = best_result["params"] -> store the best parameters to use


,learning_rate,max_depth,best_iteration,val_rmse
0,0.05,3,1999,0.005174
1,0.03,3,1999,0.006480
2,0.05,2,1998,0.008172
3,0.03,2,1999,0.008901


Best parameters:
{'learning_rate': 0.05, 'max_depth': 3}
Best iteration: 1999
Validation RMSE: 0.005174
Final best iteration: 1999
Test RMSE: 0.013080


# Rapid check

`OVERFITTING` is clearly happening here, the validation RMSE improved compared to earlier while the test RMSE got worse.

The simpler model of **6.** does not have this problem with similar RMSE.

Another point is best iteration went to the limit of n_estimators, showing the early stopping did not activate,\
validation score kept improving letting the model overfit.  

# Functions recap

---

`scaler` = ...

`scaler.fit_transform(X_train)`

`scaler.transform(X_test)`

---

```python
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components))
]) 
```

-> can then call `pipe.fit()`, `pipe.transform()` to automate the discipline (scaler -> pca -> model for example)

for PCA, n_components = int gives the number of components we keep, between 0 and 1 it gives how much variance should be explained

to access a step:

```python
pipe.named_steps['step_name'] ~ pipe.named_steps['pca']
```

to reach a parameter:
```python
pipe.set_params(step_name__parameter_name = value) ~ pipe.set_params(pca__n_components = 0.95)
```

---

`pca.explained_variance_ratio_` -> gives the list of explained variance as a fraction of total variance for each component

`pca.components_` -> gives the coefficients to transform the original column into each component 

---

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(
        n_clusters=3,
        n_init=20,
        random_state=RANDOM_STATE
    ))
]) 
```

-> recommended to use different init with KMeans 

`KMeans.predict()` gives a hard classification, `KMeans.transform()` gives distance to each cluster

`Remark`: the label does not hold economic value, we need to attribute a state after comparing them 

---

```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("gmm", GaussianMixture(
        n_components=3,
        covariance_type="full",
        n_init=10,
        random_state=RANDOM_STATE
    ))
])
```

-> `gmm.predict()` gives a hard classification, `gmm.predict_proba()` gives probabilities for each cluster

`Remark`: with KMeans distance, we do 1/dist and normalize but it is a choice, not direct probabilities such as gmm assumes

---

```python
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
) 
```

-> usually goes with hyperparameter tuning from validation. `xgb.fit()` and `xgb.predict()` as for other models

`xgb.feature_importances_` gives the importance by feature but depends on fit, OOS and time dynamics are better confirmations

with early stopping from validation not improving, it can limit the number of trees:

```python
xgb_early_stop = XGBRegressor(
    n_estimators=2000,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    early_stopping_rounds=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_early_stop.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
```

---

`tscv = TimeSeriesSplit(n_splits, test_size, gap)` -> to split in chronological order, train is expanding here

then `for fold, (train_idx, val_idx) in enumerate(tscv.split(X), start=1)`

by default select test_size = n_samples // (n_splits+1) then goes backward to get the train and test splits checking with gap

`Remark`: an appropriate complete split uses `dev and oos` separately then within each a walk-forward with `train - val - test`, tscv only has 2 out of the 3.\
For this complete setup, a homemade function is needed.

---

```python
param_grid = ParameterGrid({
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.05],
}) 
```

-> create grid of parameters for hyperparameter tuning

```python
preprocessor = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),
]) 
```

-> only preprocess to keep the model out with early stopping and grid

```python
for params in param_grid:
    prep = clone(preprocessor)

    X_train_transformed = prep.fit_transform(X_train)
    X_val_transformed = prep.transform(X_val)

    model = XGBRegressor(
        **params,
        n_estimators=2000,
        objective="reg:squarederror",
        eval_metric="rmse",
        early_stopping_rounds=50,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    model.fit(
        X_train_transformed,
        y_train,
        eval_set=[(X_val_transformed, y_val)],
        verbose=False,
    )
```

```python
tuning_results = (
    pd.DataFrame([
        {
            **result["params"],
            "best_iteration": result["best_iteration"],
            "val_rmse": result["val_rmse"],
        }
        for result in results
    ])
    .sort_values("val_rmse")
    .reset_index(drop=True)
) 
```

-> store the result and ready to inspect, especially regarding stability

```python
best_result = min(
    results,
    key=lambda x: x["val_rmse"]
)
best_params = best_result["params"] 
```

-> store the best parameters to use